# Grain_Key_Validation_evidence

## Purpose

This notebook provides reproducible evidence for the warehouse grain, key, relationship, and cardinality validation performed during the phase, (i Couldnt just aford to leave all those scripts be forgotten after i run them in shell, besides i can still use them in the future for my own reference and utilization)

The validation is executed against the project's `marketing_analytics.duckdb` database using Python and DuckDB.

The underlying SQL tests are preserved separately in:

`tests/sql/phase_5_3_grain_key_validation.sql`

This notebook serves as the execution and evidence layer for those tests, besides the fact that this is a deliberate and a more broader measure-reassignment investigation.


## Validation Scope

The validation covers:

- Expected warehouse grain
- Row-level grain preservation
- Key uniqueness
- Key completeness
- Campaign population continuity
- Fact-to-dimension relationships
- Join cardinality
- Fan-out risk
- Dimension grain
- Fact grain
- Alignment with the warehouse design


## Path to the file
##### Tree \

```text
tests/
├── sql/
│   ├── phase_5_2_data_validation.sql
│   └── phase_5_3_grain_key_validation.sql
    └── Grain_Key_Validation_evidence.ipynb
```

## Validation Evidence:

```text

TEST
├── 1. Environment & Connection
│   ├── Python environment
│   ├── DuckDB connection
│   └── marketing_analytics.duckdb
│
├── 2. Expected Grain Definition
│   ├── Raw grain
│   ├── Staging grain
│   ├── Intermediate grain
│   ├── Dimension grain
│   ├── Fact grain
│   └── campaign_id as declared business key
│
├── 3. Row Grain Validation
│   ├── Row count by layer
│   ├── Distinct campaign_id count
│   └── Duplicate excess
│
├── 4. Key Validation
│   ├── campaign_id uniqueness
│   ├── campaign_id completeness
│   └── NULL key validation
│
├── 5. Campaign Population Continuity
│   ├── RAW >> STAGING
│   ├── STAGING >> INTERMEDIATE
│   ├── INTERMEDIATE >> DIMENSION
│   └── INTERMEDIATE >> FACT
│
├── 6. Fact >> Dimension Relationship
│   ├── Total fact rows
│   ├── Matched fact campaigns
│   └── Unmatched fact campaigns
│
├── 7. Join Cardinality
│   ├── Joined row count
│   ├── Distinct fact campaigns
│   ├── Distinct dimension campaigns
│   └── Duplicate excess after join
│
├── 8. Fan-Out Risk
│   ├── Multiple dimension matches per campaign
│   └── Campaigns producing >1 joined record
│
├── 9. Dimension Grain
│   ├── One row per campaign
│   ├── Duplicate campaign records
│   └── Conflicting descriptive attributes
│
├── 10. Fact Grain
│   ├── One row per campaign
│   ├── Duplicate campaign records
│   └── Conflicting measure values
│
├── 11. Grain Preservation
│   ├── RAW >> STAGING
│   ├── STAGING >> INTERMEDIATE
│   ├── INTERMEDIATE >> DIMENSION
│   └── INTERMEDIATE >> FACT
│
├── 12. Warehouse Design Alignment
│   ├── Compare implementation to Phase 2 design
│   ├── Confirm current two-model mart is sufficient
│   └── Confirm no unnecessary dimensional expansion
│
└── 13. Final Assessment
    ├── PASS / FAIL
    ├── Findings
    ├── Engineering risks
    ├── Assumptions
    ├── Limitations
    └── Design decision

```

what i did was run all these in shell by simply running

python
this will deliberetely push you to run python scripts in shell

```
(.venv) PS C:\Users\JayPogi\D************\D******\Sk******\Data********\G*****\marketing-analytics-engineering-platform> python
Python 3.13.3 (tags/v3.13.3:6280bb5, Apr  8 2025, 14:47:33) [MSC v.1943 64 bit (AMD64)] on win32
Type "help", "copyright", "credits" or "license" for more information.
>>>
>>>

```

i run everything and got my results from here, this is just a documentation of the tests i did.

### Environment & (Duck) Connection



In [ ]:
from pathlib import Path
import duckdb

# Resolve the project root from the notebook's working directory.
PROJECT_ROOT = Path.cwd().parents[1]

# Connect to the project warehouse in read-only mode.
DB_PATH = PROJECT_ROOT / "marketing_analytics.duckdb"
con = duckdb.connect(str(DB_PATH), read_only=True)

print("Project root:")
print(PROJECT_ROOT)

print("\nDatabase:")
print(DB_PATH)

print("\nDatabase exists:")
print(DB_PATH.exists())

print("\nDuckDB connection:")
print(con.execute("PRAGMA database_list").fetchall())

print("\nWarehouse models:")
print(con.execute("SHOW TABLES").fetchall())



Project root:
c:\Users\JayPogi\Documents\Documents\Skul\Data_Science\GIT\marketing-analytics-engineering-platform

Database:
c:\Users\JayPogi\Documents\Documents\Skul\Data_Science\GIT\marketing-analytics-engineering-platform\marketing_analytics.duckdb

Database exists:
True

DuckDB connection:
[(20657, 'marketing_analytics', 'C:\\Users\\JayPogi\\Documents\\Documents\\Skul\\Data_Science\\GIT\\marketing-analytics-engineering-platform\\marketing_analytics.duckdb')]

Warehouse models:
[('dim_campaign',), ('fct_campaign_performance',), ('int_campaign_performance',), ('stg_campaign_performance',)]


In [4]:
raw_count = con.execute("""
    SELECT COUNT(*)
    FROM raw.marketing_campaign
""").fetchone()[0]

fact_count = con.execute("""
    SELECT COUNT(*)
    FROM main.fct_campaign_performance
""").fetchone()[0]

print("Raw campaigns:", raw_count)
print("Fact campaigns:", fact_count)

Raw campaigns: 1831
Fact campaigns: 1831


### Expected Grain Definition

The current warehouse is designed at a campaign-level grain.

The declared business key is `campaign_id`.

Expected grain by layer:

| Layer | Model | Expected Grain | Business Key |
|---|---|---|---|
| Raw | `raw.marketing_campaign` | One row per campaign | `campaign_id` |
| Staging | `stg_campaign_performance` | One row per campaign | `campaign_id` |
| Intermediate | `int_campaign_performance` | One row per campaign | `campaign_id` |
| Dimension | `dim_campaign` | One row per campaign | `campaign_id` |
| Fact | `fct_campaign_performance` | One row per campaign | `campaign_id` |

The validation below tests whether the implemented warehouse preserves this declared grain.

### Row Grain Validation

The warehouse is expected to maintain one row per `campaign_id` across the raw, staging, intermediate, dimension, and fact layers.

This section compares total row counts, distinct campaign counts, and duplicate excess for each layer.

A duplicate excess value of zero means:

            `total rows - distinct campaign_id = 0`

which supports the declared one-row-per-campaign grain >>> Does each individual layer have the correct grain?

In [5]:
models = {
    "RAW": "raw.marketing_campaign",
    "STAGING": "main.stg_campaign_performance",
    "INTERMEDIATE": "main.int_campaign_performance",
    "DIMENSION": "main.dim_campaign",
    "FACT": "main.fct_campaign_performance",
}

row_grain_results = []

for layer, table in models.items():
    result = con.execute(f"""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT campaign_id) AS distinct_campaign_ids,
            COUNT(*) - COUNT(DISTINCT campaign_id) AS duplicate_excess
        FROM {table}
    """).fetchone()

    row_grain_results.append(
        (layer, table, result[0], result[1], result[2])
    )

print("Layer | Total Rows | Distinct Campaign IDs | Duplicate Excess")
print("-" * 70)

for row in row_grain_results:
    print(
        f"{row[0]:<14} | "
        f"{row[2]:>10} | "
        f"{row[3]:>21} | "
        f"{row[4]:>16}"
    )

Layer | Total Rows | Distinct Campaign IDs | Duplicate Excess
----------------------------------------------------------------------
RAW            |       1831 |                  1831 |                0
STAGING        |       1831 |                  1831 |                0
INTERMEDIATE   |       1831 |                  1831 |                0
DIMENSION      |       1831 |                  1831 |                0
FACT           |       1831 |                  1831 |                0


In [6]:
### QUALITY CHECK:

grain_pass = all(
    total_rows == distinct_ids and duplicate_excess == 0
    for _, _, total_rows, distinct_ids, duplicate_excess in row_grain_results
)

print("Row grain validation:", "PASS" if grain_pass else "FAIL")

Row grain validation: PASS


### Key Validation

The declared business key for the current warehouse is `campaign_id`.

This section explicitly validates key uniqueness and completeness across the raw, staging, intermediate, dimension, and fact layers.

A valid campaign-level key must satisfy:

- `campaign_id` is not NULL
- each `campaign_id` occurs at most once per layer
- the number of non-NULL distinct keys matches the expected campaign population

In [7]:
# ID Uniqueness Validation

key_uniqueness_results = []

for layer, table in models.items():
    duplicate_keys = con.execute(f"""
        SELECT
            campaign_id,
            COUNT(*) AS record_count
        FROM {table}
        GROUP BY campaign_id
        HAVING COUNT(*) > 1
        ORDER BY record_count DESC, campaign_id
    """).fetchall()

    key_uniqueness_results.append(
        (layer, table, len(duplicate_keys), duplicate_keys)
    )

print("Campaign ID uniqueness validation")
print("-" * 70)

for layer, table, duplicate_count, duplicates in key_uniqueness_results:
    print(
        f"{layer:<14} | Duplicate campaign IDs: {duplicate_count}"
    )

Campaign ID uniqueness validation
----------------------------------------------------------------------
RAW            | Duplicate campaign IDs: 0
STAGING        | Duplicate campaign IDs: 0
INTERMEDIATE   | Duplicate campaign IDs: 0
DIMENSION      | Duplicate campaign IDs: 0
FACT           | Duplicate campaign IDs: 0


In [8]:
# ID completness Validation

key_completeness_results = []

for layer, table in models.items():
    total_rows, null_keys, distinct_non_null_keys = con.execute(f"""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(*) - COUNT(campaign_id) AS null_campaign_ids,
            COUNT(DISTINCT campaign_id) AS distinct_non_null_campaign_ids
        FROM {table}
    """).fetchone()

    key_completeness_results.append(
        (
            layer,
            table,
            total_rows,
            null_keys,
            distinct_non_null_keys
        )
    )

print("Campaign ID completeness validation")
print("-" * 90)
print(
    "Layer          | Total Rows | NULL Keys | Distinct Non-NULL Keys"
)
print("-" * 90)

for row in key_completeness_results:
    print(
        f"{row[0]:<14} | "
        f"{row[2]:>10} | "
        f"{row[3]:>9} | "
        f"{row[4]:>22}"
    )

Campaign ID completeness validation
------------------------------------------------------------------------------------------
Layer          | Total Rows | NULL Keys | Distinct Non-NULL Keys
------------------------------------------------------------------------------------------
RAW            |       1831 |         0 |                   1831
STAGING        |       1831 |         0 |                   1831
INTERMEDIATE   |       1831 |         0 |                   1831
DIMENSION      |       1831 |         0 |                   1831
FACT           |       1831 |         0 |                   1831


In [9]:
### QUALITY CHECK:

key_uniqueness_pass = all(
    duplicate_count == 0
    for _, _, duplicate_count, _ in key_uniqueness_results
)

key_completeness_pass = all(
    null_keys == 0 and distinct_non_null_keys == total_rows
    for _, _, total_rows, null_keys, distinct_non_null_keys
    in key_completeness_results
)

key_validation_pass = key_uniqueness_pass and key_completeness_pass

print(
    "Key validation:",
    "PASS" if key_validation_pass else "FAIL"
)

Key validation: PASS


## Campaign Population Continuity

This section validates that the campaign population is preserved across each warehouse transformation layer.

The expected behavior is:

    RAW >> STAGING >> INTERMEDIATE >> DIMENSION
                                    >> FACT

For each transition, the validation checks both directions:

- campaigns present in the upstream layer but missing downstream
- campaigns present downstream but not present upstream

A valid transformation should produce zero missing and zero unexpected campaign IDs.

In [10]:
population_checks = [
    ("RAW → STAGING", "raw.marketing_campaign", "main.stg_campaign_performance"),
    ("STAGING → INTERMEDIATE", "main.stg_campaign_performance", "main.int_campaign_performance"),
    ("INTERMEDIATE → DIMENSION", "main.int_campaign_performance", "main.dim_campaign"),
    ("INTERMEDIATE → FACT", "main.int_campaign_performance", "main.fct_campaign_performance"),
]

population_results = []

for transition, upstream, downstream in population_checks:

    upstream_missing_downstream = con.execute(f"""
        SELECT COUNT(*)
        FROM (
            SELECT campaign_id
            FROM {upstream}

            EXCEPT

            SELECT campaign_id
            FROM {downstream}
        )
    """).fetchone()[0]

    downstream_missing_upstream = con.execute(f"""
        SELECT COUNT(*)
        FROM (
            SELECT campaign_id
            FROM {downstream}

            EXCEPT

            SELECT campaign_id
            FROM {upstream}
        )
    """).fetchone()[0]

    population_results.append(
        (
            transition,
            upstream_missing_downstream,
            downstream_missing_upstream
        )
    )

print("Campaign population continuity")
print("-" * 80)
print(
    "Transition                    | "
    "Missing Downstream | "
    "Unexpected Downstream"
)
print("-" * 80)

for transition, missing_downstream, unexpected_downstream in population_results:
    print(
        f"{transition:<29} | "
        f"{missing_downstream:>17} | "
        f"{unexpected_downstream:>20}"
    )

Campaign population continuity
--------------------------------------------------------------------------------
Transition                    | Missing Downstream | Unexpected Downstream
--------------------------------------------------------------------------------
RAW → STAGING                 |                 0 |                    0
STAGING → INTERMEDIATE        |                 0 |                    0
INTERMEDIATE → DIMENSION      |                 0 |                    0
INTERMEDIATE → FACT           |                 0 |                    0


In [11]:
#### Quality Check:

population_pass = all(
    missing_downstream == 0 and unexpected_downstream == 0
    for _, missing_downstream, unexpected_downstream
    in population_results
)

print(
    "Campaign population continuity:",
    "PASS" if population_pass else "FAIL"
)

Campaign population continuity: PASS


### Fact & Dimension Relationship

The fact table uses `campaign_id` as its key and is expected to relate to `dim_campaign` through that key.

This validation confirms that every fact record has a corresponding dimension record.

The expected relationship at the current warehouse grain is:

`fct_campaign_performance.campaign_id`
>> `dim_campaign.campaign_id`

Because both models are currently one row per campaign, the expected relationship is one-to-one.

In [12]:
fact_dimension_result = con.execute("""
    SELECT
        COUNT(*) AS total_fact_rows,
        COUNT(d.campaign_id) AS matched_fact_campaigns,
        COUNT(*) - COUNT(d.campaign_id) AS unmatched_fact_campaigns
    FROM main.fct_campaign_performance f
    LEFT JOIN main.dim_campaign d
        ON f.campaign_id = d.campaign_id
""").fetchone()

print("Fact → Dimension relationship")
print("-" * 60)
print(f"Total fact rows:          {fact_dimension_result[0]}")
print(f"Matched fact campaigns:   {fact_dimension_result[1]}")
print(f"Unmatched fact campaigns: {fact_dimension_result[2]}")

Fact → Dimension relationship
------------------------------------------------------------
Total fact rows:          1831
Matched fact campaigns:   1831
Unmatched fact campaigns: 0


In [13]:
# quality check for fact → dimension relationship

fact_dimension_pass = (
    fact_dimension_result[0] == fact_dimension_result[1]
    and fact_dimension_result[2] == 0
)

print(
    "Fact >> Dimension relationship:",
    "PASS" if fact_dimension_pass else "FAIL"
)

Fact >> Dimension relationship: PASS


### Join Cardinality

This section validates the cardinality of the fact-to-dimension join.

The current warehouse design expects a one-to-one relationship at the campaign grain:

`fct_campaign_performance.campaign_id`
>>> `dim_campaign.campaign_id`

The joined result should therefore preserve the fact row count.

A joined row count greater than the fact population would indicate potential fan-out or duplicate dimension matches.

In [14]:
join_cardinality_result = con.execute("""
    SELECT
        COUNT(*) AS joined_rows,
        COUNT(DISTINCT f.campaign_id) AS distinct_fact_campaigns,
        COUNT(DISTINCT d.campaign_id) AS distinct_dimension_campaigns,
        COUNT(*) - COUNT(DISTINCT f.campaign_id) AS duplicate_excess
    FROM main.fct_campaign_performance f
    INNER JOIN main.dim_campaign d
        ON f.campaign_id = d.campaign_id
""").fetchone()

print("Fact → Dimension join cardinality")
print("-" * 70)
print(f"Joined rows:                  {join_cardinality_result[0]}")
print(f"Distinct fact campaigns:      {join_cardinality_result[1]}")
print(f"Distinct dimension campaigns: {join_cardinality_result[2]}")
print(f"Duplicate excess:             {join_cardinality_result[3]}")

Fact → Dimension join cardinality
----------------------------------------------------------------------
Joined rows:                  1831
Distinct fact campaigns:      1831
Distinct dimension campaigns: 1831
Duplicate excess:             0


In [15]:
### QUALITY CHECK:

join_cardinality_pass = (
    join_cardinality_result[0] == 1831
    and join_cardinality_result[1] == 1831
    and join_cardinality_result[2] == 1831
    and join_cardinality_result[3] == 0
)

print(
    "Join cardinality:",
    "PASS" if join_cardinality_pass else "FAIL"
)

Join cardinality: PASS


### Fan-Out Risk

A fan-out occurs when one fact record matches multiple dimension records during a join.

At the current campaign grain, each `campaign_id` should appear exactly once in `dim_campaign`.

This section identifies any campaign that could produce multiple joined records.

Expected result: zero campaigns with multiple dimension matches.

In [16]:
fanout_results = con.execute("""
    SELECT
        f.campaign_id,
        COUNT(*) AS dimension_matches
    FROM main.fct_campaign_performance f
    INNER JOIN main.dim_campaign d
        ON f.campaign_id = d.campaign_id
    GROUP BY f.campaign_id
    HAVING COUNT(*) > 1
    ORDER BY dimension_matches DESC, f.campaign_id
""").fetchall()

print("Campaigns with multiple dimension matches:")
print(fanout_results)
print(f"\nFan-out campaign count: {len(fanout_results)}")

Campaigns with multiple dimension matches:
[]

Fan-out campaign count: 0


In [17]:
#### Quality CHeck:

fanout_pass = len(fanout_results) == 0

print(
    "Fan-out risk validation:",
    "PASS" if fanout_pass else "FAIL"
)

Fan-out risk validation: PASS


###  Dimension Grain

The dimension table is currently defined at the grain of one row per campaign.

Each `campaign_id` should therefore have exactly one descriptive record containing a consistent set of campaign attributes.

This section checks for:

- duplicate campaign records
- conflicting descriptive attributes within the same campaign

In [18]:
dimension_grain_results = con.execute("""
    SELECT
        campaign_id,
        COUNT(*) AS record_count,
        COUNT(DISTINCT campaign_name) AS campaign_name_values,
        COUNT(DISTINCT start_date) AS start_date_values,
        COUNT(DISTINCT end_date) AS end_date_values,
        COUNT(DISTINCT channel) AS channel_values,
        COUNT(DISTINCT region) AS region_values,
        COUNT(DISTINCT target_audience) AS target_audience_values,
        COUNT(DISTINCT product_category) AS product_category_values,
        COUNT(DISTINCT device) AS device_values,
        COUNT(DISTINCT year) AS year_values
    FROM main.dim_campaign
    GROUP BY campaign_id
    HAVING
        COUNT(*) > 1
        OR COUNT(DISTINCT campaign_name) > 1
        OR COUNT(DISTINCT start_date) > 1
        OR COUNT(DISTINCT end_date) > 1
        OR COUNT(DISTINCT channel) > 1
        OR COUNT(DISTINCT region) > 1
        OR COUNT(DISTINCT target_audience) > 1
        OR COUNT(DISTINCT product_category) > 1
        OR COUNT(DISTINCT device) > 1
        OR COUNT(DISTINCT year) > 1
    ORDER BY campaign_id
""").fetchall()

print("Dimension grain violations:")
print(dimension_grain_results)
print(f"\nViolation count: {len(dimension_grain_results)}")

Dimension grain violations:
[]

Violation count: 0


In [19]:
#### Quality Check:

dimension_grain_pass = len(dimension_grain_results) == 0

print(
    "Dimension grain validation:",
    "PASS" if dimension_grain_pass else "FAIL"
)

Dimension grain validation: PASS


### Fact Grain

The fact table is currently defined at the grain of one row per campaign.

Each `campaign_id` should therefore have exactly one record containing one consistent set of campaign performance measures.

This section checks for:

- duplicate campaign records
- conflicting measure values within the same campaign

In [20]:

fact_grain_results = con.execute("""
    SELECT
        campaign_id,
        COUNT(*) AS record_count,
        COUNT(DISTINCT impressions) AS impressions_values,
        COUNT(DISTINCT clicks) AS clicks_values,
        COUNT(DISTINCT conversions) AS conversions_values,
        COUNT(DISTINCT spend_usd) AS spend_values,
        COUNT(DISTINCT revenue_usd) AS revenue_values
    FROM main.fct_campaign_performance
    GROUP BY campaign_id
    HAVING
        COUNT(*) > 1
        OR COUNT(DISTINCT impressions) > 1
        OR COUNT(DISTINCT clicks) > 1
        OR COUNT(DISTINCT conversions) > 1
        OR COUNT(DISTINCT spend_usd) > 1
        OR COUNT(DISTINCT revenue_usd) > 1
    ORDER BY campaign_id
""").fetchall()

print("Fact grain violations:")
print(fact_grain_results)
print(f"\nViolation count: {len(fact_grain_results)}")

Fact grain violations:
[]

Violation count: 0


In [21]:
#### Quality heck

fact_grain_pass = len(fact_grain_results) == 0

print(
    "Fact grain validation:",
    "PASS" if fact_grain_pass else "FAIL"
)

Fact grain validation: PASS


### Grain Preservation

The warehouse is designed around a one-row-per-campaign grain.

This section summarizes whether each transformation preserves that grain across the implemented pipeline.

The validation considers:

- total row count
- distinct `campaign_id` count
- duplicate excess

A transformation preserves the declared grain when:

`total rows = distinct campaign_id count`

and:

`duplicate excess = 0`

across each layer.

In [22]:
grain_preservation_results = []

for layer, table in models.items():
    total_rows, distinct_ids, duplicate_excess = con.execute(f"""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT campaign_id) AS distinct_campaign_ids,
            COUNT(*) - COUNT(DISTINCT campaign_id) AS duplicate_excess
        FROM {table}
    """).fetchone()

    grain_preservation_results.append(
        (
            layer,
            total_rows,
            distinct_ids,
            duplicate_excess
        )
    )

print("Grain preservation across warehouse layers")
print("-" * 75)
print(
    "Layer          | Total Rows | Distinct IDs | Duplicate Excess"
)
print("-" * 75)

for layer, total_rows, distinct_ids, duplicate_excess in grain_preservation_results:
    print(
        f"{layer:<14} | "
        f"{total_rows:>10} | "
        f"{distinct_ids:>12} | "
        f"{duplicate_excess:>16}"
    )

Grain preservation across warehouse layers
---------------------------------------------------------------------------
Layer          | Total Rows | Distinct IDs | Duplicate Excess
---------------------------------------------------------------------------
RAW            |       1831 |         1831 |                0
STAGING        |       1831 |         1831 |                0
INTERMEDIATE   |       1831 |         1831 |                0
DIMENSION      |       1831 |         1831 |                0
FACT           |       1831 |         1831 |                0


In [23]:
#### QUALITY CHECK:

grain_preservation_pass = all(
    total_rows == distinct_ids and duplicate_excess == 0
    for _, total_rows, distinct_ids, duplicate_excess
    in grain_preservation_results
)

print(
    "Grain preservation:",
    "PASS" if grain_preservation_pass else "FAIL"
)

Grain preservation: PASS


### Warehouse Design Alignment

The implemented warehouse is compared against the architecture and warehouse design established during Phase 2.

The current design intentionally uses the smallest useful dimensional model required by the current business requirements and source grain.

Current mart structure:

    dim_campaign
          │
    fct_campaign_performance

Both models operate at one row per campaign using `campaign_id`.

Additional candidate dimensions such as channel, region, target audience, product, device, and date were evaluated but deferred because the current business requirements and source grain do not require independent dimensional entities.


In [24]:
design_alignment = {
    "Grain": {
        "expected": "1 row per campaign",
        "implemented": "1 row per campaign",
        "status": "PASS",
    },
    "Business key": {
        "expected": "campaign_id",
        "implemented": "campaign_id",
        "status": "PASS",
    },
    "Fact model": {
        "expected": "Campaign performance measures",
        "implemented": "impressions, clicks, conversions, spend_usd, revenue_usd",
        "status": "PASS",
    },
    "Dimension model": {
        "expected": "Campaign descriptive attributes",
        "implemented": "campaign_name, dates, channel, region, audience, product, device, year",
        "status": "PASS",
    },
    "Mart scope": {
        "expected": "Smallest useful model for current requirements",
        "implemented": "dim_campaign + fct_campaign_performance",
        "status": "PASS",
    },
    "Speculative dimensions": {
        "expected": "Evaluate and defer unless independently required",
        "implemented": "No unnecessary dimensional expansion",
        "status": "PASS",
    },
}

print("Warehouse design alignment")
print("-" * 100)

for check, details in design_alignment.items():
    print(f"{check}")
    print(f"  Expected:     {details['expected']}")
    print(f"  Implemented:  {details['implemented']}")
    print(f"  Status:       {details['status']}")
    print()

Warehouse design alignment
----------------------------------------------------------------------------------------------------
Grain
  Expected:     1 row per campaign
  Implemented:  1 row per campaign
  Status:       PASS

Business key
  Expected:     campaign_id
  Implemented:  campaign_id
  Status:       PASS

Fact model
  Expected:     Campaign performance measures
  Implemented:  impressions, clicks, conversions, spend_usd, revenue_usd
  Status:       PASS

Dimension model
  Expected:     Campaign descriptive attributes
  Implemented:  campaign_name, dates, channel, region, audience, product, device, year
  Status:       PASS

Mart scope
  Expected:     Smallest useful model for current requirements
  Implemented:  dim_campaign + fct_campaign_performance
  Status:       PASS

Speculative dimensions
  Expected:     Evaluate and defer unless independently required
  Implemented:  No unnecessary dimensional expansion
  Status:       PASS



In [25]:
#### Quality check:

design_alignment_pass = all(
    details["status"] == "PASS"
    for details in design_alignment.values()
)

print(
    "Warehouse design alignment:",
    "PASS" if design_alignment_pass else "FAIL"
)

Warehouse design alignment: PASS


### Final Assessment

The Phase 5.3 Grain & Key Validation confirms that the implemented warehouse preserves the declared one-row-per-campaign grain and maintains key and relationship integrity across the current transformation layers.

#### Validation Results

- Expected grain: PASS
- Row grain validation: PASS
- Key uniqueness: PASS
- Key completeness: PASS
- Campaign population continuity: PASS
- Fact-to-dimension relationship: PASS
- Join cardinality: PASS
- Fan-out risk: PASS
- Dimension grain: PASS
- Fact grain: PASS
- Grain preservation: PASS
- Warehouse design alignment: PASS

#### Engineering Findings

The current warehouse maintains `campaign_id` as the business key across the implemented layers.

The fact and dimension models preserve the one-row-per-campaign grain, and the fact-to-dimension relationship produces no orphan records or fan-out.

#### Assumptions and Limitations

The current warehouse operates at campaign-level grain. Additional dimensions were evaluated but intentionally deferred because the current business requirements and source grain do not require independent dimensional entities.

The intermediate model contains a dataset-specific measure reconstruction rule that assigns the largest, middle, and smallest source values to impressions, clicks, and conversions respectively. This rule is documented separately and should be revisited when a source with reliable semantic validation or source metadata becomes available.

#### Design Decision

Retain the current:

`dim_campaign + fct_campaign_performance`

mart structure.

No additional dimensions are introduced solely to increase model complexity or portfolio scope.

In [ ]:
validation_summary = {
    "Expected grain": grain_pass,
    "Key validation": key_validation_pass,
    "Campaign population continuity": population_pass,
    "Fact → Dimension relationship": fact_dimension_pass,
    "Join cardinality": join_cardinality_pass,
    "Fan-out risk": fanout_pass,
    "Dimension grain": dimension_grain_pass,
    "Fact grain": fact_grain_pass,
    "Grain preservation": grain_preservation_pass,
    "Warehouse design alignment": design_alignment_pass,
}

print("Phase 5.3 Validation Summary")
print("=" * 70)

for check, result in validation_summary.items():
    print(f"{check:<40} {'PASS' if result else 'FAIL'}")

phase_5_3_pass = all(validation_summary.values())

print("=" * 70)
print(
    "PHASE 5.3 FINAL STATUS:",
    "PASS" if phase_5_3_pass else "FAIL"
)

Phase 5.3 Validation Summary
Expected grain                           PASS
Key validation                           PASS
Campaign population continuity           PASS
Fact → Dimension relationship            PASS
Join cardinality                         PASS
Fan-out risk                             PASS
Dimension grain                          PASS
Fact grain                               PASS
Grain preservation                       PASS
Warehouse design alignment               PASS
PHASE 5.3 FINAL STATUS: PASS


: 